### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="indian_liver_patient_dataset",
    dataset_year="2012",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D02C",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/225/ilpd+indian+liver+patient+dataset.zip && unzip ilpd+indian+liver+patient+dataset.zip && rm ilpd+indian+liver+patient+dataset.zip
mkdir -p local-data-warehouse/indian_liver_patient_dataset && mv "Indian Liver Patient Dataset (ILPD).csv" local-data-warehouse/indian_liver_patient_dataset/
""",
    # References
    academic_reference_bibtex="""@article{ramana2012critical,
  title={A critical comparative study of liver patients from USA and INDIA: an exploratory analysis},
  author={Ramana, Bendi Venkata and Babu, M Surendra Prasad and Venkateswarlu, NB},
  journal={International Journal of Computer Science Issues (IJCSI)},
  volume={9},
  number={3},
  pages={506},
  year={2012},
  publisher={International Journal of Computer Science Issues (IJCSI)}
}
""",
    academic_reference_bibtex_key="ramana2012critical",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We keep the data as is as it does not require any further cleaning.
Note, the data contains a few (n=13) duplicates, that are likely naturally occurring duplicates given the features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Selector",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Selector",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    dataset_mold.path / "Indian Liver Patient Dataset (ILPD).csv",
    header=None,
    names=["Age", "Gender", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/G", "Selector"]
)
print("Loaded data shape:", df.shape)

as_cat_type = ["Gender", "Selector"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (583, 11)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 583
Columns: 11
Use sampling: False (sample size: 583)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Alkphos', 'Sgot', 'Sgpt', 'TB', 'DB', 'Age', 'A/G', 'TP', 'ALB', 'Gender']
Rows remaining as candidates after top-10 filter: 26 (of 583)

#### Duplicate Report
Total duplicate rows: 13 (2.23% of dataset)
Duplicate rows ignoring target: 13 (2.23% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G,Selector
0,19,Male,1.4,0.8,178,13,26,8.0,4.6,1.30,2
1,12,Male,1.0,0.2,719,157,108,7.2,3.7,1.00,1
2,60,Male,5.7,2.8,214,412,850,7.3,3.2,0.78,1
3,42,Female,0.5,0.1,162,155,108,8.1,4.0,0.90,1
4,40,Male,14.5,6.4,358,50,75,5.7,2.1,0.50,1


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Gender,category,0.0,0.00,2.0,"Male, Female"
1,Selector,category,0.0,0.00,2.0,"1, 2"
2,A/G,float64,4.0,0.69,69.0,"1.0, 0.8, 0.9, 0.7, 1.1, 1.2, 0.6, 0.5, 1.3, 1.4"
3,TB,float64,0.0,0.00,113.0,"0.8, 0.7, 0.9, 0.6, 1.0, 1.1, 1.8, 1.4, 1.3, 1.7"
4,DB,float64,0.0,0.00,80.0,"0.2, 0.1, 0.3, 0.8, 0.4, 0.5, 0.6, 1.0, 1.3, 0.7"
5,TP,float64,0.0,0.00,58.0,"7.0, 6.0, 6.8, 6.9, 6.2, 7.1, 7.2, 8.0, 6.1, 7.3"
6,ALB,float64,0.0,0.00,40.0,"3.0, 4.0, 2.9, 3.1, 3.2, 3.9, 2.5, 2.7, 3.5, 3.4"
7,Age,int64,0.0,0.00,72.0,"60, 45, 50, 42, 38, 32, 48, 55, 65, 40"
8,Alkphos,int64,0.0,0.00,263.0,"215, 198, 298, 180, 195, 190, 145, 182, 158, 165"
9,Sgpt,int64,0.0,0.00,152.0,"25, 20, 22, 21, 28, 18, 30, 48, 15, 24"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,583.0,44.746141,16.189833,4.0,90.0
TB,583.0,3.298799,6.209522,0.4,75.0
DB,583.0,1.486106,2.808498,0.1,19.7
Alkphos,583.0,290.576329,242.937989,63.0,2110.0
Sgpt,583.0,80.713551,182.620356,10.0,2000.0
Sgot,583.0,109.910806,288.918529,10.0,4929.0
TP,583.0,6.483190,1.085451,2.7,9.6
ALB,583.0,3.141852,0.795519,0.9,5.5
A/G,579.0,0.947064,0.319592,0.3,2.8


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                      
Gender   1       Male    441  75.64
         2     Female    142  24.36
Selector 1          1    416  71.36
         2          2    167  28.64

In [8]:
# Target Distribution
target_df

,count,pct
Selector,,
1,416,71.36
2,167,28.64


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to indian_liver_patient_dataset/019d736d-ec03-75de-b8e9-075fcdde7993


019d736d-ec03-75de-b8e9-075fcdde7993
5927755df2ad2cebfef10357310f3f5de7dafd7f92b855368b9e276451405db6
